In [ ]:
!pip install transformers
!pip install tensorflow #--upgrade
!pip install keras #--upgrade

In [ ]:
import tensorflow as tf
import numpy as np 
import pandas as pd
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from keras.layers import Bidirectional
from sklearn.model_selection import train_test_split
from keras.utils.np_utils import to_categorical

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from keras.callbacks import EarlyStopping
from keras.layers import Dropout
import re
from nltk.corpus import stopwords
from nltk import word_tokenize
from bs4 import BeautifulSoup
import plotly.graph_objs as go
import cufflinks
from IPython.core.interactiveshell import InteractiveShell
import plotly.figure_factory as ff
InteractiveShell.ast_node_interactivity = 'all'
from plotly.offline import iplot
cufflinks.go_offline()
cufflinks.set_config_file(world_readable=True, theme='pearl')
from nltk.corpus import stopwords
from keras.layers import Bidirectional
from tensorflow.keras.optimizers import Adam

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

Found GPU at: /device:GPU:0


In [ ]:
train_df = pd.read_csv("train.tsv", sep="\t")
val_df = pd.read_csv("val_empty.tsv", sep="\t")
test_df = pd.read_csv("test-no_labels.tsv", sep="\t")

#First Tryy

In [ ]:
class ModuleLayer(tf.keras.layers.Layer):

  @staticmethod
  def __init_session__(session):
    tf.keras.backend.set_session(session)

In [ ]:
# Define constants
MAX_NB_WORDS = 50000
MAX_SEQUENCE_LENGTH = 250
EMBEDDING_DIM = 150
epochs = 10
batch_size = 64

# Create tokenizer
tokenizer = Tokenizer(num_words=MAX_NB_WORDS, filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~', lower=True)
tokenizer.fit_on_texts(train_df['text'].values)
word_index = tokenizer.word_index

# Define model architecture
model = Sequential()
model.add(ModuleLayer())
model.add(Embedding(len(word_index) + 1, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH))
model.add(SpatialDropout1D(0.5))
model.add(Bidirectional(LSTM(150)))
model.add(Dense(4, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model for each column
for col in ['masks_stance', 'masks_argument', 'quarantine_stance', 'quarantine_argument', 'vaccines_stance', 'vaccines_argument']:
  Y = pd.get_dummies(train_df[col]).values
  X = tokenizer.texts_to_sequences(train_df['text'].values)
  X = pad_sequences(X, maxlen=MAX_SEQUENCE_LENGTH)

  X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.1, random_state=42)

  model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=epochs, batch_size=batch_size)

  x = tokenizer.texts_to_sequences(test_df['text'].values)
  x = pad_sequences(x, maxlen=MAX_SEQUENCE_LENGTH)

  predict_proba = model.predict(x)
  predict = np.argmax(predict_proba, axis=1)
  test_df[col] = predict - 1


Epoch 1/10
95/95 [==============================] - 38s 228ms/step - loss: 0.9737 - accuracy: 0.6397 - val_loss: 0.5394 - val_accuracy: 0.8065
Epoch 2/10
95/95 [==============================] - 14s 151ms/step - loss: 0.4935 - accuracy: 0.7949 - val_loss: 0.4530 - val_accuracy: 0.8214
Epoch 3/10
95/95 [==============================] - 9s 96ms/step - loss: 0.4084 - accuracy: 0.8202 - val_loss: 0.4624 - val_accuracy: 0.7872
Epoch 4/10
95/95 [==============================] - 9s 91ms/step - loss: 0.2970 - accuracy: 0.8864 - val_loss: 0.4557 - val_accuracy: 0.8080
Epoch 5/10
95/95 [==============================] - 6s 60ms/step - loss: 0.1849 - accuracy: 0.9333 - val_loss: 0.5484 - val_accuracy: 0.7932
Epoch 6/10
95/95 [==============================] - 5s 57ms/step - loss: 0.1095 - accuracy: 0.9656 - val_loss: 0.5915 - val_accuracy: 0.7946
Epoch 7/10
95/95 [==============================] - 6s 69ms/step - loss: 0.0681 - accuracy: 0.9793 - val_loss: 0.7212 - val_accuracy: 0.7872
Epoch 8/1

44/44 [==============================] - 1s 11ms/step
Epoch 1/10
95/95 [==============================] - 4s 48ms/step - loss: 0.2896 - accuracy: 0.8945 - val_loss: 0.3237 - val_accuracy: 0.8899
Epoch 2/10
95/95 [==============================] - 4s 42ms/step - loss: 0.1689 - accuracy: 0.9398 - val_loss: 0.3064 - val_accuracy: 0.9048
Epoch 3/10
95/95 [==============================] - 5s 51ms/step - loss: 0.1142 - accuracy: 0.9629 - val_loss: 0.3190 - val_accuracy: 0.8958
Epoch 4/10
95/95 [==============================] - 4s 37ms/step - loss: 0.0656 - accuracy: 0.9811 - val_loss: 0.3751 - val_accuracy: 0.8914
Epoch 5/10
95/95 [==============================] - 4s 41ms/step - loss: 0.0418 - accuracy: 0.9858 - val_loss: 0.3776 - val_accuracy: 0.8929
Epoch 6/10
95/95 [==============================] - 4s 43ms/step - loss: 0.0262 - accuracy: 0.9932 - val_loss: 0.4056 - val_accuracy: 0.9003
Epoch 7/10
95/95 [==============================] - 4s 38ms/step - loss: 0.0128 - accuracy: 0.9964 -

44/44 [==============================] - 0s 8ms/step
Epoch 1/10
95/95 [==============================] - 4s 38ms/step - loss: 1.1459 - accuracy: 0.6779 - val_loss: 0.4783 - val_accuracy: 0.8244
Epoch 2/10
95/95 [==============================] - 4s 41ms/step - loss: 0.3050 - accuracy: 0.8916 - val_loss: 0.3093 - val_accuracy: 0.8839
Epoch 3/10
95/95 [==============================] - 4s 42ms/step - loss: 0.1975 - accuracy: 0.9310 - val_loss: 0.3363 - val_accuracy: 0.8839
Epoch 4/10
95/95 [==============================] - 4s 38ms/step - loss: 0.1219 - accuracy: 0.9573 - val_loss: 0.3990 - val_accuracy: 0.8601
Epoch 5/10
95/95 [==============================] - 3s 36ms/step - loss: 0.0693 - accuracy: 0.9778 - val_loss: 0.4270 - val_accuracy: 0.8557
Epoch 6/10
95/95 [==============================] - 3s 37ms/step - loss: 0.0516 - accuracy: 0.9836 - val_loss: 0.4641 - val_accuracy: 0.8438
Epoch 7/10
95/95 [==============================] - 4s 38ms/step - loss: 0.0269 - accuracy: 0.9911 - 

44/44 [==============================] - 0s 11ms/step
Epoch 1/10
95/95 [==============================] - 3s 33ms/step - loss: 0.1937 - accuracy: 0.9393 - val_loss: 0.2519 - val_accuracy: 0.9345
Epoch 2/10
95/95 [==============================] - 4s 37ms/step - loss: 0.1104 - accuracy: 0.9664 - val_loss: 0.2559 - val_accuracy: 0.9286
Epoch 3/10
95/95 [==============================] - 3s 33ms/step - loss: 0.0759 - accuracy: 0.9765 - val_loss: 0.2782 - val_accuracy: 0.9301
Epoch 4/10
95/95 [==============================] - 4s 37ms/step - loss: 0.0527 - accuracy: 0.9835 - val_loss: 0.2906 - val_accuracy: 0.9256
Epoch 5/10
95/95 [==============================] - 4s 37ms/step - loss: 0.0330 - accuracy: 0.9907 - val_loss: 0.3143 - val_accuracy: 0.9226
Epoch 6/10
95/95 [==============================] - 4s 37ms/step - loss: 0.0230 - accuracy: 0.9940 - val_loss: 0.3362 - val_accuracy: 0.9182
Epoch 7/10
95/95 [==============================] - 4s 40ms/step - loss: 0.0110 - accuracy: 0.9975 -

44/44 [==============================] - 0s 8ms/step
Epoch 1/10
95/95 [==============================] - 3s 36ms/step - loss: 0.8787 - accuracy: 0.7633 - val_loss: 0.3114 - val_accuracy: 0.8571
Epoch 2/10
95/95 [==============================] - 4s 44ms/step - loss: 0.2659 - accuracy: 0.8878 - val_loss: 0.2750 - val_accuracy: 0.8824
Epoch 3/10
95/95 [==============================] - 4s 37ms/step - loss: 0.1567 - accuracy: 0.9424 - val_loss: 0.3453 - val_accuracy: 0.8586
Epoch 4/10
95/95 [==============================] - 4s 37ms/step - loss: 0.0790 - accuracy: 0.9714 - val_loss: 0.3475 - val_accuracy: 0.8735
Epoch 5/10
95/95 [==============================] - 3s 36ms/step - loss: 0.0350 - accuracy: 0.9881 - val_loss: 0.4102 - val_accuracy: 0.8690
Epoch 6/10
95/95 [==============================] - 4s 43ms/step - loss: 0.0185 - accuracy: 0.9952 - val_loss: 0.4699 - val_accuracy: 0.8646
Epoch 7/10
95/95 [==============================] - 3s 34ms/step - loss: 0.0096 - accuracy: 0.9985 - 

44/44 [==============================] - 0s 9ms/step
Epoch 1/10
95/95 [==============================] - 3s 34ms/step - loss: 0.1808 - accuracy: 0.9375 - val_loss: 0.2244 - val_accuracy: 0.9092
Epoch 2/10
95/95 [==============================] - 3s 36ms/step - loss: 0.0938 - accuracy: 0.9656 - val_loss: 0.2454 - val_accuracy: 0.8988
Epoch 3/10
95/95 [==============================] - 3s 35ms/step - loss: 0.0607 - accuracy: 0.9801 - val_loss: 0.2851 - val_accuracy: 0.9092
Epoch 4/10
95/95 [==============================] - 4s 42ms/step - loss: 0.0322 - accuracy: 0.9901 - val_loss: 0.3463 - val_accuracy: 0.9048
Epoch 5/10
95/95 [==============================] - 4s 38ms/step - loss: 0.0217 - accuracy: 0.9942 - val_loss: 0.3804 - val_accuracy: 0.9092
Epoch 6/10
95/95 [==============================] - 3s 34ms/step - loss: 0.0121 - accuracy: 0.9965 - val_loss: 0.4165 - val_accuracy: 0.9062
Epoch 7/10
95/95 [==============================] - 3s 32ms/step - loss: 0.0045 - accuracy: 0.9995 - 

44/44 [==============================] - 0s 8ms/step


In [ ]:
test_df.to_csv('result9.tsv', index = False, sep='\t')
!zip result9.zip result9.tsv

  adding: result9.tsv (deflated 72%)


#Second try

https://colab.research.google.com/drive/1rePNC8taIpc4aREwcvO49mxe2AxCZ17Z?usp=sharing